In [17]:
import numpy as np
import allel
import csv
from tqdm import tqdm
from collections import Counter, defaultdict
from typing import List
import pandas as pd


In [18]:
country_to_region = pd.read_csv("country_to_region.tsv", delimiter='\t')
country_to_region

,Country,Region
0,Bangladesh,South Asia
1,Benin,Western Africa
2,Burkina Faso,Western Africa
3,Cambodia,Southeast Asia
4,Cameroon,Western Africa
5,Colombia,South America
6,Cote d'Ivoire,Western Africa
7,Democratic Republic of the Congo,Central Africa
8,Ethiopia,Eastern Africa
9,Gabon,Central Africa


In [19]:
metadata = pd.read_csv("julian_meta.csv")
metadata

,sample_id,sample,sra_run,country,site,year,Study,collaborator
0,ERR1081237,FP0008-C,ERR1081237,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN
1,ERR1081238,FP0009-C,ERR1081238,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN
2,ERR2889621,FP0010-CW,ERR2889621,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN
3,ERR2889624,FP0011-CW,ERR2889624,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN
4,ERR2889627,FP0012-CW,ERR2889627,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN
...,...,...,...,...,...,...,...,...
20859,ERR3486353,SPT43399,ERR3486353,NaN,NaN,NaN,1132-PF-K1000G-DBS-KE-BEJON,NaN
20860,ERR3486368,SPT43400,ERR3486368,NaN,NaN,NaN,1132-PF-K1000G-DBS-KE-BEJON,NaN
20861,ERR3486366,SPT43401,ERR3486366,NaN,NaN,NaN,1132-PF-K1000G-DBS-KE-BEJON,NaN
20862,ERR3486400,SPT43403,ERR3486400,Kenya,NaN,2018.0,1132-PF-K1000G-DBS-KE-BEJON,NaN


In [20]:
metadata = pd.merge(metadata, country_to_region, left_on='country', right_on='Country', how='left')
metadata

,sample_id,sample,sra_run,country,site,year,Study,collaborator,Country,Region
0,ERR1081237,FP0008-C,ERR1081237,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN,Mauritania,Western Africa
1,ERR1081238,FP0009-C,ERR1081238,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN,Mauritania,Western Africa
2,ERR2889621,FP0010-CW,ERR2889621,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN,Mauritania,Western Africa
3,ERR2889624,FP0011-CW,ERR2889624,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN,Mauritania,Western Africa
4,ERR2889627,FP0012-CW,ERR2889627,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN,Mauritania,Western Africa
...,...,...,...,...,...,...,...,...,...,...
20859,ERR3486353,SPT43399,ERR3486353,NaN,NaN,NaN,1132-PF-K1000G-DBS-KE-BEJON,NaN,NaN,NaN
20860,ERR3486368,SPT43400,ERR3486368,NaN,NaN,NaN,1132-PF-K1000G-DBS-KE-BEJON,NaN,NaN,NaN
20861,ERR3486366,SPT43401,ERR3486366,NaN,NaN,NaN,1132-PF-K1000G-DBS-KE-BEJON,NaN,NaN,NaN
20862,ERR3486400,SPT43403,ERR3486400,Kenya,NaN,2018.0,1132-PF-K1000G-DBS-KE-BEJON,NaN,Kenya,Eastern Africa


In [ ]:
# input_file = 'merged.filtered.vcf.gz'
input_file = 'variants.vcf.gz'
callset = allel.read_vcf(input_file)

In [32]:
rows = []
for i in tqdm(range(len(callset['variants/POS']))):
    mean_gt = np.nanmean([x[0] for x in callset['calldata/GT'][i] if x[0]!=-1])
    row = [x[1] if x[1]!=-1 else np.nan for x in callset['calldata/GT'][i]]
    rows.append(row)

100%|██████████| 56/56 [00:00<00:00, 114.30it/s]


In [33]:
sample_vectors = []
for i in tqdm(range(len(callset['samples']))):
    sample_vectors.append([x[i] for x in rows])
sample_vectors = np.array(sample_vectors)
sample_vectors.shape

100%|██████████| 19843/19843 [00:00<00:00, 198456.69it/s]


(19843, 56)

In [35]:
def sample2pop(sample: str, metadata: pd.DataFrame) -> str:
    row = metadata[metadata['sample'] == sample]
    return row.iloc[0]['Region']


y = [sample2pop(s, metadata) for s in callset['samples']]



In [36]:
X = np.array(sample_vectors)

In [37]:
from sklearn.ensemble import HistGradientBoostingClassifier
clf = HistGradientBoostingClassifier(max_iter=100).fit(X, y)
clf.score(X, y)

0.933780174368795

In [41]:
def sample2country(sample: str, metadata: pd.DataFrame) -> str:
    row = metadata[metadata['sample'] == sample]
    return row.iloc[0]['country']

In [42]:
y_country = np.array([sample2country(s, metadata) for s in callset['samples']])
clf_country = HistGradientBoostingClassifier(max_iter=100).fit(X, y_country)

In [43]:
clf_country.score(X, y_country)

0.5004787582522804

In [44]:
import json
geojson = json.load(open("world.geojson"))


In [48]:
df = pd.DataFrame({'country':clf_country.classes_,'probability':clf_country.predict_proba(X)[1000]})
subgeojson = {'type': 'FeatureCollection', 'features': [x for x in geojson['features'] if x['properties']['ADMIN'] in df['country'].values]}

In [49]:
import plotly.express as px
import plotly.graph_objects as go

In [50]:
fig = go.Figure(data=go.Choropleth(
    locations=df['country'],locationmode="country names",
    z=df['probability'], colorscale='Reds',
    marker_line_color='darkgray',
    marker_line_width=0.5,
    )
)
fig.update_layout(
    title_text='Geographic source probability',
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='equirectangular'
    ),
)
fig.update_layout(
    margin=dict(l=20, r=20, t=60, b=20),
)
fig.show()

/tmp/ipykernel_3672991/2156714765.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = go.Figure(data=go.Choropleth(
